In [1]:
import sys
from pathlib import Path
print(sys.executable)
assert Path(sys.prefix).resolve() == Path('/home/miakhlov/course-notebooks/.venv').resolve()

/home/miakhlov/course-notebooks/.venv/bin/python


# Операция Химера: первый проверенный результат
Откройте notebook из папки practice или корня проекта. Установите requirements.txt в окружение выбранного ядра. Данные искусственные: из них нельзя делать вывод об активности. В Colab сначала загрузите chemistry.py, stepik_tasks.py и оба CSV в рабочую папку; установки выполняются только для открытых учебных данных.

In [2]:
import sys
from pathlib import Path
import csv
import math
import rdkit
print(sys.executable, sys.version, rdkit.__version__)
candidates = [Path.cwd(), Path.cwd() / 'Stepik_course' / 'practice']
practice = next((p for p in candidates if (p / 'chemistry.py').exists()), None)
if practice is None:
    raise FileNotFoundError('Откройте notebook из practice или корня проекта')
sys.path.insert(0, str(practice))
from chemistry import standardize, descriptors, screen, curate_activity, similarity, optimize_conformer
from stepik_tasks import tanimoto, enrichment, z_prime

/home/miakhlov/course-notebooks/.venv/bin/python 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0] 2026.09.1


## 1. Сначала спрогнозируйте
Какая часть натрий ацетата останется? Сохранится ли ковалентный Cl в хлорбензоле? Запишите прогноз до запуска. Нейтральный parent не означает, что измерения соли и кислоты взаимозаменяемы.

In [3]:
salt = standardize('CC(=O)[O-].[Na+]')
print(salt)
print(standardize('Clc1ccccc1'))
print(descriptors(salt.parent_smiles))
assert salt.parent_smiles == standardize('CC(=O)O').parent_smiles
assert 'Cl' in standardize('Clc1ccccc1').parent_smiles
assert math.isclose(descriptors('CCO')['MolWt'], 46.069, abs_tol=0.01)

Structure(raw_smiles='CC(=O)[O-].[Na+]', parent_smiles='CC(=O)O', policy='teaching-parent-v1;neutralize;tautomer=False')
Structure(raw_smiles='Clc1ccccc1', parent_smiles='Clc1ccccc1', policy='teaching-parent-v1;neutralize;tautomer=False')
{'MolWt': 60.05200000000001, 'MolLogP': 0.09089999999999993, 'TPSA': 37.3, 'HBD': 1, 'HBA': 1, 'ro5_violations': 0}


[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Running LargestFragmentChooser
[12:55:20] Fragment: CC(=O)[O-]
[12:55:20] New largest fragment: CC(=O)[O-] (7)
[12:55:20] Fragment: [Na+]
[12:55:20] Running Uncharger
[12:55:20] Removed negative charge.
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Running LargestFragmentChooser
[12:55:20] Running Uncharger
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normal

## 2. Предупреждение и доказательство
Предскажите статусы этанола, катехола и неверной записи. Затем объясните, почему eligible не означает безопасен или активен.

In [4]:
with (practice / 'molecules.csv').open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
results = [{'id': row['id'], **screen(row['smiles'])} for row in rows]
for row in results:
    print(row['id'], row['status'], row['reason'])
assert len(results) == len(rows)
assert next(r for r in results if r['id'] == 'M06')['status'] == 'review'
assert next(r for r in results if r['id'] == 'M09')['status'] == 'invalid'

[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Running LargestFragmentChooser
[12:55:20] Running Uncharger
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Running LargestFragmentChooser
[12:55:20] Running Uncharger
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Normalizer
[12:55:20] Initializing MetalDisconnector
[12:55:20] Running MetalDisconnector
[12:55:20] Initializing Normalizer
[12:55:20] Running Norma

M01 eligible No listed alert; activity remains unknown
M02 eligible No listed alert; activity remains unknown
M03 eligible No listed alert; activity remains unknown
M04 eligible No listed alert; activity remains unknown
M05 eligible No listed alert; activity remains unknown
M06 review PAINS:catechol_A(92)
M07 eligible No listed alert; activity remains unknown
M08 invalid Invalid SMILES
M09 invalid SMILES must be a nonempty string
M10 eligible No listed alert; activity remains unknown
M11 eligible No listed alert; activity remains unknown
M12 eligible No listed alert; activity remains unknown


[12:55:21] Initializing MetalDisconnector
[12:55:21] Running MetalDisconnector
[12:55:21] Initializing Normalizer
[12:55:21] Running Normalizer
[12:55:21] Initializing MetalDisconnector
[12:55:21] Running MetalDisconnector
[12:55:21] Initializing Normalizer
[12:55:21] Running Normalizer
[12:55:21] Running LargestFragmentChooser
[12:55:21] Running Uncharger


## 3. Данные активности
Объясните каждое исключение. Числа вымышлены. R01 и R02 не являются независимыми подтверждениями.

In [5]:
with (practice / 'activities.csv').open(encoding='utf-8') as handle:
    activities = [curate_activity(row) for row in csv.DictReader(handle)]
for row in activities:
    print(row['record_id'], row['status'], row.get('pIC50'), row['reason'])
assert [r['record_id'] for r in activities if r['status'] == 'included'] == ['R01', 'R02']

R01 included 7.0 teaching rules satisfied
R02 included 7.0 teaching rules satisfied
R03 excluded None relation
R04 excluded None endpoint
R05 excluded None target_id
R06 excluded None assay_id
R07 excluded None value_or_unit
R08 excluded None value_or_unit


## 4. Представления и геометрия
Почему сходство 1 не доказывает измеренную активность? Почему оптимизация одного конформера не доказывает глобальный минимум?

In [6]:
print('Танимото:', tanimoto({1,2,3}, {2,3,4}))
print('Morgan:', similarity('CCO', 'OCC'))
print('Z prime:', z_prime(100, 10, 5, 5))
print('EF:', enrichment(1000, 20, 50, 5))
print('Конформер:', optimize_conformer('CCCC'))

Танимото: 0.5
Morgan: 1.0
Z prime: 0.6666666666666667
EF: 5.0
Конформер: {'converged': True, 'status_code': 0, 'energy_kcal_mol': -4.29376426829433, 'seed': 42, 'n_conformers': 1}


## 5. Самостоятельная часть
Добавьте новый валидный SMILES, пустую строку и структуру с ковалентным Br. Проверьте журнал. Напишите три предложения: какой вопрос решён; какой вывод пока недопустим; какой следующий контроль нужен. Для настоящего QSPR используйте отдельный train_esol.py и требования requirements-ml.txt, а не искусственные CSV.